In [8]:
#Importar liberias
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, cheby1, firwin, filtfilt, lfilter #importarcion de los filtros
import pandas as pd

# Seleccion de archivos y filtros a aplicar.
folder_path = r"C:\Users\tulio\Documents\LAB5"  
output_img_path = "./graficas_ecg"
os.makedirs(output_img_path, exist_ok=True)

#En este caso seleccione los filtros Butter, Cheby, Balckman y Hamming.
filenames = {
    "Basal 1": {
        "file": "Basal_D2_1.txt",
        "iir": "butter",
        "fir": "hamming"
    },
    "Respiración": {
        "file": "Respiracion5_D2_1.txt",
        "iir": "cheby",
        "fir": "blackman"
    },
    "Post-ejercicio": {
        "file": "After_ejercicio_D2_1.txt",
        "iir": "butter",
        "fir": "blackman"
    }
}

COLUMN_INDEX = -1
SAMPLING_RATE = 1000 # Frecuencia de muestreo en Hz
LOW_CUT = 0.5  #valores minimos y maximos de corte de pasabanda
HIGH_CUT = 40 

# Creando los filtros en funciones
def butter_filter(data, fs):
    nyq = 0.5 * fs
    b, a = butter(4, [LOW_CUT / nyq, HIGH_CUT / nyq], btype='band')
    return filtfilt(b, a, data)

def cheby_filter(data, fs):
    nyq = 0.5 * fs
    b, a = cheby1(4, 1, [LOW_CUT / nyq, HIGH_CUT / nyq], btype='band')
    return filtfilt(b, a, data)

def fir_filter(data, fs, window):
    nyq = 0.5 * fs
    taps = firwin(101, [LOW_CUT / nyq, HIGH_CUT / nyq], pass_zero=False, window=window) # valor de N = 100
    return lfilter(taps, 1.0, data)

# Quitar el encabezado de los archivos de texto
def read_opensignals_txt(filepath):
    with open(filepath, 'r') as file:
        lines = file.readlines()
    data_start = 0
    for i, line in enumerate(lines):
        if line.strip() == "# EndOfHeader":
            data_start = i + 1
            break
    data_lines = lines[data_start:]
    data = [list(map(int, line.strip().split('\t'))) for line in data_lines if line.strip()]
    return np.array(data)

# Creacion de las graficas con sus respectivos nombres de encabezado
md_lines = ["# Comparación de Filtros ECG\n", "| Señal | Sin Filtro | Filtro IIR | Filtro FIR |", "|-------|-------|-------------|------------|"]
# aplicacion de for para cada señal
for label, config in filenames.items():
    filepath = os.path.join(folder_path, config["file"])
    data = read_opensignals_txt(filepath)

    if data.shape[1] < abs(COLUMN_INDEX):
        print(f"Advertencia: el archivo {filepath} no tiene suficientes columnas.")
        continue
    # Aplicacion de los filtros a las señales
    raw = data[:, COLUMN_INDEX]
    iir = butter_filter(raw, SAMPLING_RATE) if config["iir"] == "butter" else cheby_filter(raw, SAMPLING_RATE)
    fir = fir_filter(raw, SAMPLING_RATE, config["fir"])
    
    time = np.arange(len(raw)) / SAMPLING_RATE #se crea el tiempo que queremos analizar

    # Creacion de los graficos sin filtrar
    fig1, ax1 = plt.subplots(figsize=(12, 3))
    ax1.plot(time, raw, color='gray')
    ax1.set_title(f"{label} - Crudo")
    ax1.set_xlabel("Tiempo (s)")
    ax1.set_ylabel("Amplitud")
    ax1.grid(True)
    fig1.tight_layout()
    img_crudo = os.path.join(output_img_path, f"{label}_crudo.png")
    fig1.savefig(img_crudo)
    plt.close(fig1)

    # Creacion de los graficos con Filtro IRR
    fig2, ax2 = plt.subplots(figsize=(12, 3))
    ax2.plot(time, iir, color='blue')
    ax2.set_title(f"{label} - Filtro IIR ({config['iir'].capitalize()})")
    ax2.set_xlabel("Tiempo (s)")
    ax2.set_ylabel("Amplitud")
    ax2.grid(True)
    fig2.tight_layout()
    img_iir = os.path.join(output_img_path, f"{label}_iir.png")
    fig2.savefig(img_iir)
    plt.close(fig2)

    # Creacion de los graficos con Filtro FIR
    fig3, ax3 = plt.subplots(figsize=(12, 3))
    ax3.plot(time, fir, color='green')
    ax3.set_title(f"{label} - Filtro FIR ({config['fir'].capitalize()})")
    ax3.set_xlabel("Tiempo (s)")
    ax3.set_ylabel("Amplitud")
    ax3.grid(True)
    fig3.tight_layout()
    img_fir = os.path.join(output_img_path, f"{label}_fir.png")
    fig3.savefig(img_fir)
    plt.close(fig3)

    # Creacion de los archivos con sus respectivos nombres en mi carpeta
    md_crudo = os.path.join("graficas_ecg", f"{label}_crudo.png")
    md_iir = os.path.join("graficas_ecg", f"{label}_iir.png")
    md_fir = os.path.join("graficas_ecg", f"{label}_fir.png")
    md_lines.append(f"| {label} | ![]({md_crudo}) | ![]({md_iir}) | ![]({md_fir}) |")

# guardarlo en mi pc con el formato Markdown 
with open("resumen_resultados.md", "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))

print(" Creacion de los graficos realizados 'graficas_ecg'")


 Creacion de los graficos realizados 'graficas_ecg'
